# 04b — Model improvement experiments (Temporal approach)

Tests four hypotheses raised while reviewing the SHAP results in `04_models.ipynb`, against the actual CatBoost model (Option B, time split). This notebook is deliberately separate from `04_models.ipynb` — it is exploratory, not part of the stable pipeline, and meant to be run on its own branch.

**Result up front, not buried at the end:** of the three things tested, only one held up. The other two are reported anyway, with the numbers that ruled them out.

| # | Hypothesis | Outcome |
|---|---|---|
| 1 | Mental-health features hide multicollinearity beyond what full-dataset VIF caught | **Not confirmed** — max VIF 3.64 within the block |
| 2 | Test/Val gap is partly a COVID-period effect, not just time-distance decay | **Confirmed** |
| 3 | Dropping lowest-importance features improves generalization | **Not confirmed** — Val and Test R² both dropped |

> **Note added after `04_models.ipynb` was extended with per-country SARIMAX/Prophet:** the hypotheses below were tested against the panel CatBoost model specifically, before the time-series models existed. They remain valid as stated — the COVID-period finding in particular is a property of the *years themselves*, not of any one model — but do not assume they transfer unchanged to SARIMAX/Prophet, which learn each country's own trajectory rather than a pooled cross-country regression. Re-testing these same four hypotheses against the time-series models would be a reasonable next step, not something already covered here.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostRegressor

from src import (
    ID_COLS,
    TARGET,
    HEALTH_RELATED_FEATURES,
    build_predictor_list,
    temporal_split,
    compute_vif,
    evaluate_model,
    metrics_by_period,
    save_figure,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
np.random.seed(42)

df_development = pd.read_parquet("../data/processed/df_development.parquet")
predictor_features = build_predictor_list(df_development, ID_COLS, TARGET)
print(
    f"df_development: {df_development.shape[0]} rows | {len(predictor_features)} predictors"
)

fig_prefix = "04b_"

## Shared setup (Option B — same split as 04_models.ipynb)

Rebuilds the same train/test/val split and scaling used in `04_models.ipynb`, so every variant below is compared on identical data.
Only tests `CatBoost` because it was the winner in the models notebook.

In [ ]:
df_train_B, df_test_B, df_val_B, train_years, test_years, val_years = temporal_split(
    df_development
)
y_train_B, y_test_B, y_val_B = df_train_B[TARGET], df_test_B[TARGET], df_val_B[TARGET]


def scale_split(feature_list):
    scaler = RobustScaler()
    X_train = pd.DataFrame(
        scaler.fit_transform(df_train_B[feature_list]),
        columns=feature_list,
        index=df_train_B.index,
    )
    X_test = pd.DataFrame(
        scaler.transform(df_test_B[feature_list]),
        columns=feature_list,
        index=df_test_B.index,
    )
    X_val = pd.DataFrame(
        scaler.transform(df_val_B[feature_list]),
        columns=feature_list,
        index=df_val_B.index,
    )
    return X_train, X_test, X_val


X_train_B, X_test_B, X_val_B = scale_split(predictor_features)

# Baseline: exact hyperparameters GridSearchCV selected for CatBoost in 04_models.ipynb,
# refit here directly (no search) purely as the reference point for the comparisons below.
baseline_model = CatBoostRegressor(
    depth=4,
    iterations=400,
    l2_leaf_reg=7,
    learning_rate=0.1,
    random_state=42,
    verbose=0,
)
trained_baseline = {
    "name": "CatBoost",
    "best_estimator": baseline_model.fit(X_train_B, y_train_B),
    "best_params": {
        "depth": 4,
        "iterations": 400,
        "l2_leaf_reg": 7,
        "learning_rate": 0.1,
    },
    "cv_rmse": None,
    "time_s": None,
}
eval_baseline_test = evaluate_model(trained_baseline, X_test_B, y_test_B, "Test")
eval_baseline_val = evaluate_model(trained_baseline, X_val_B, y_val_B, "Val")

## Hypothesis 1 — Hidden multicollinearity within the mental-health block

`02_eda.ipynb`'s VIF check ran on the full predictor set and only flagged `Eating disorders` (VIF 12.3, already dropped). The concern here was narrower: mental-health prevalence causes come from the same underlying study (GBD) and might share enough variance with *each other* specifically to distort SHAP's attribution, even without triggering the full-dataset threshold. Re-running VIF restricted to just this block checks that directly.

In [ ]:
mental_health_features = [f for f in HEALTH_RELATED_FEATURES if f != TARGET]
vif_mental_health = compute_vif(df_development, mental_health_features)
vif_mental_health

**Result: not confirmed.** The highest VIF within the block is 3.64 (`Attention-deficit/hyperactivity disorder`) — below the moderate-concern threshold of 5. There is no meaningful hidden collinearity here; `Alcohol use disorders` dominating the SHAP ranking is not an artifact of correlated features splitting credit unevenly. 

## Hypothesis 2 — Is the Test/Val gap a COVID-period effect?

Splits Option B's Validation years (2018–2021) into Pre-COVID (2018–2019) and COVID (2020–2021), and computes metrics separately for each, using `metrics_by_period()`.

In [ ]:
periods = {"Pre-COVID (2018-2019)": [2018, 2019], "COVID (2020-2021)": [2020, 2021]}

period_metrics = metrics_by_period(
    eval_baseline_val["actuals"],
    eval_baseline_val["predictions"],
    df_val_B["Year"].reset_index(drop=True),
    periods,
)
period_metrics

**Result: confirmed.** R² drops from 0.72 (Pre-COVID) to 0.47 (COVID) within the same Validation set, using the same model. A meaningful share of the overall Val R² (0.605) reported in `04_models.ipynb` is attributable to a two-year structural shock the model — trained on 2000–2014 — had no way to anticipate.

## Hypothesis 3 — Dropping the lowest-importance features improves generalization

Drops the three lowest-ranked features from both the SHAP and CatBoost native importance rankings computed in `04_models.ipynb` (`Health expenditure (% GDP)`, `Unemployment rate (%)`, `Conduct disorder`) and retrains with identical hyperparameters, to isolate the effect of the feature set alone.

In [ ]:
LOW_IMPORTANCE_FEATURES = [
    "Health expenditure (% GDP)",
    "Unemployment rate (%)",
    "Conduct disorder",
]
trimmed_features = build_predictor_list(
    df_development, ID_COLS, TARGET, extra_exclude=LOW_IMPORTANCE_FEATURES
)
print(f"Predictors: {len(predictor_features)} -> {len(trimmed_features)}")

X_train_trim, X_test_trim, X_val_trim = scale_split(trimmed_features)

trimmed_model = CatBoostRegressor(
    depth=4,
    iterations=400,
    l2_leaf_reg=7,
    learning_rate=0.1,
    random_state=42,
    verbose=0,
)
trained_trimmed = {
    "name": "CatBoost (trimmed)",
    "best_estimator": trimmed_model.fit(X_train_trim, y_train_B),
    "best_params": {},
    "cv_rmse": None,
    "time_s": None,
}
eval_trimmed_test = evaluate_model(trained_trimmed, X_test_trim, y_test_B, "Test")
eval_trimmed_val = evaluate_model(trained_trimmed, X_val_trim, y_val_B, "Val")

print(
    f"\nBaseline  — Test R²: {eval_baseline_test['r2']} | Val R²: {eval_baseline_val['r2']}"
)
print(
    f"Trimmed   — Test R²: {eval_trimmed_test['r2']} | Val R²: {eval_trimmed_val['r2']}"
)

**Result: not confirmed — it makes things slightly worse.** Test R² drops from 0.875 to 0.848, Val R² from 0.605 to 0.577. With 594 rows and 18 predictors, this dataset is not in the regime where CatBoost benefits from feature pruning — the "noise" hypothesis behind this experiment does not hold. The three dropped features were carrying some real signal despite their low individual SHAP ranking, likely through interactions rather than main effects. Keep the full feature set.

## Summary

| Variant | Test R² | Val R² | Change vs baseline |
|---|---|---|---|
| Baseline (as in `04_models.ipynb`) | 0.875 | 0.605 | — |
| Trimmed features | 0.848 | 0.577 | Worse |

Out of the three things tested, the only one worth carrying back is the COVID-period breakdown (Hypothesis 2).